# Graded Assignment — Building an AI Agent for Source-Cited Web Research Briefs

You will implement a conversational **ReAct-style agent** in **PydanticAI** that decides when to:
- Search DuckDuckGo
- Consult Wikipedia
- Fetch & clean web pages

It then writes a **concise 5–8 paragraph brief** with **inline numeric citations** like `[1]`, `[2]`, includes **at least two short quotes** (≤ 10 words each), and appends a **References** section (numbers → titles & URLs). You’ll **persist tool logs** for grading.

**API:** This assignment uses **OpenRouter**. Enter your OpenRouter API key when prompted; do not hard-code it.

**Deliverables (auto-checked):**
- `brief.md` and `report.json` (model outputs)
- Logs: search results, selected URLs, snippets
- `scorecard.json` with simple metrics


# Read-Only Instructions

Install (locally/Codio):
```
pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv
```
Environment:
- Set `OPENROUTER_API_KEY` as an environment variable.
- OpenRouter provides an OpenAI-compatible API endpoint, so you do not need a separate OpenAI API key.
- The notebook will prompt you for the key securely when it runs in Colab.

In [ ]:
# ! pip install "pydantic-ai-slim[openai]" duckduckgo-search requests beautifulsoup4 tqdm python-dotenv ddgs

In [ ]:
# OpenRouter Environment
import os
from getpass import getpass

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass('Enter your OpenRouter API key: ')

OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OPENROUTER_MODEL = 'openai/gpt-4o-mini'

# OpenRouter is OpenAI-compatible, but uses its own API key and endpoint.


Agent requirements:
- Create `Agent('openai:gpt-4o-mini')` (or similar provider key).
- Register tools (`@agent.tool` / `@agent.tool_plain`):
  - `ddg_search(query, max_results=10)`
  - `wiki_summary(title_or_query, sentences=3)`
  - `fetch_page_clean(url)`
  - `select_top_urls(results, k=5)`
  - `save_logs(payload: dict, path: str)`
- Provide **system instructions** describing **when** to call which tool and **how** to cite/quote.
- Support **multi-turn** using PydanticAI `message_history` (`result.new_messages()`).

Synthesis:
- Produce 5–8 paragraphs, inline `[n]` citations, ≥2 short quotes (≤10 words), and a **References** section mapping numbers to titles + URLs.
- Save artifacts: `brief.md`, `report.json`, and logs (search results, selected URLs, snippets).

Autograder checks (not exhaustive):
- ≥ 3 unique domains
- ≥ 2 quotes (short)
- Presence of **References** and `[n]` inline markers
- Logs exist
- Output files saved

In [ ]:
# Imports (Read-Only)
from typing import Any, Dict, List
import json, re, os, pathlib
from urllib.parse import urlparse

# Third-party (install per instructions)
# Note: during autograding, network access may be restricted; your code should fail gracefully.
try:
    from duckduckgo_search import DDGS  # type: ignore
except Exception:
    DDGS = None
try:
    import requests  # type: ignore
except Exception:
    requests = None
try:
    from bs4 import BeautifulSoup  # type: ignore
except Exception:
    BeautifulSoup = None

# PydanticAI agent
try:
    from pydantic_ai import Agent
    from pydantic_ai.models.openai import OpenAIModel
    from pydantic_ai.providers.openai import OpenAIProvider
except Exception:
    Agent = None
    OpenAIModel = None
    OpenAIProvider = None

# OpenRouter-compatible PydanticAI model. Use this model when creating the Agent.
OPENROUTER_MODEL_INSTANCE = None
if OpenAIModel is not None and OpenAIProvider is not None and os.environ.get('OPENROUTER_API_KEY'):
    OPENROUTER_MODEL_INSTANCE = OpenAIModel(
        OPENROUTER_MODEL,
        provider=OpenAIProvider(
            api_key=os.environ['OPENROUTER_API_KEY'],
            base_url=OPENROUTER_BASE_URL,
        ),
    )

DATA_DIR = pathlib.Path('.').resolve()
ARTIFACTS_DIR = DATA_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Q1. Implement the research tools

Implement:
- `ddg_search(query: str, max_results: int = 10) -> List[Dict]`
- `wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]`
- `fetch_page_clean(url: str) -> Dict[str, str]`
- `select_top_urls(results: List[Dict], k: int = 5) -> List[str]`
- `save_logs(payload: Dict[str, Any], path: str) -> str`

### Please write your answer below

In [ ]:
# Q1 — Solution
# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
def wiki_summary(title_or_query: str, sentences: int = 3) -> Dict[str, str]:
# YOUR CODE HERE
raise NotImplementedError()


In [ ]:
def fetch_page_clean(url: str) -> Dict[str, str]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def select_top_urls(results: List[Dict[str, Any]], k: int = 5) -> List[str]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def save_logs(payload: Dict[str, Any], path: str) -> str:
# YOUR CODE HERE
raise NotImplementedError()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `ddg_search`,`wiki_summary`,`fetch_page_clean`,`select_top_urls`,`save_logs`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q1 — Public Tests
# BEGIN PUBLIC TESTS
res = ddg_search("site:example.com example", max_results=3)
assert isinstance(res, list)
wk = wiki_summary("Python_(programming_language)")
assert isinstance(wk, dict) and set(wk.keys()) >= {"title","url","summary"}
doc = fetch_page_clean("https://example.com")
assert isinstance(doc, dict) and set(doc.keys()) >= {"title","url","text"}
sel = select_top_urls([{"url":"https://en.wikipedia.org/wiki/Test"},{"url":"https://example.com"}], k=1)
assert isinstance(sel, list) and len(sel) >= 1
fp = save_logs({"ok":True}, "q1_log.json")
assert fp.endswith("q1_log.json")
# END PUBLIC TESTS

In [ ]:
# Q1 — Hidden Tests

# Q2. Define the Agent and register tools

Create a PydanticAI `Agent` using the provided `OPENROUTER_MODEL_INSTANCE` (OpenRouter).  
Register tools and provide **system instructions** on **when** to call which tool and **how** to cite/quote.

**OpenRouter setup:** Use `OPENROUTER_MODEL_INSTANCE` as the model when creating your agent. Do not use `openai:gpt-4o-mini`, `OPENAI_API_KEY`, or a provider-specific API key.


### Please write your answer below

In [ ]:
# Q2 — Solution
# YOUR CODE HERE
raise NotImplementedError()


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q2 — Public Tests
# BEGIN PUBLIC TESTS
assert 'SYSTEM_INSTRUCTIONS' in globals()
assert 'agent' in globals()
for t in ['ddg_search_tool','wiki_summary_tool','fetch_page_clean_tool','select_top_urls_tool','save_logs_tool']:
    assert t in globals(), f"Missing tool {t}"
# END PUBLIC TESTS

In [ ]:
# Q2 — Hidden Tests

# Q3. Conversational ReAct: run topic → tools → synthesis

Implement `run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]` that:
- Starts a run with the user topic.
- Lets the model call tools.
- Supports multi-turn via `message_history = result.new_messages()`.
- Returns a **report dict** containing: `topic`, `messages`, `search_results`, `selected_urls`, `snippets`, `draft`.
Also save `report.json` via `save_logs`.

### Please write your answer below

In [ ]:
# Q3 — Solution
import os, asyncio, inspect
from typing import Any, Dict

def run_agent(topic: str, rounds: int = 1) -> Dict[str, Any]:
# YOUR CODE HERE
raise NotImplementedError()


---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_agent`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q3 — Public Tests
# BEGIN PUBLIC TESTS
assert callable(run_agent)
# END PUBLIC TESTS

In [ ]:
# Q3 — Hidden Tests

# Q4. Synthesis & Output

Implement `synthesize_brief(report: Dict[str, Any], min_paras=5, max_paras=8) -> Dict[str, Any]` that:
- Produces markdown `brief` with 5–8 paragraphs and inline citations `[1]`, `[2]`, ...
- Ensures at least **two short quotes** (≤ 10 words each) are present.
- Builds a `references` list mapping numbers to `{title, url}`.
- Saves `brief.md` and returns report with `brief`, `references`.

### Please write your answer below

In [ ]:
# Q4 — Solution
def synthesize_brief(report: Dict[str, Any], min_paras: int = 5, max_paras: int = 8) -> Dict[str, Any]:
# YOUR CODE HERE
raise NotImplementedError()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `synthesize_brief`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q4 — Public Tests
# BEGIN PUBLIC TESTS
dummy = {"draft": "Para one.\n\nPara two.", "selected_urls": ["https://example.com/a","https://en.wikipedia.org/wiki/AI","https://gov.example.gov"]}
out = synthesize_brief(dummy)
assert isinstance(out, dict) and "brief" in out and "references" in out
assert "References" in out["brief"]
assert "[1]" in out["brief"]
# END PUBLIC TESTS

In [ ]:
# Q4 — Hidden Tests

# Q5. Pipeline Runner & Evaluation

Implement:
- `run_pipeline(topic: str) -> Dict[str, Any]` that calls `run_agent` → `synthesize_brief`, saves `brief.md` & `report.json`.
- `evaluate_outputs(brief_path: str = 'artifacts/brief.md', report_path: str = 'artifacts/report.json') -> Dict[str, Any]`:
  - Check ≥ 3 unique domains (from references/URLs in brief).
  - Check ≥ 2 short quotes (≤ 10 words).
  - Check presence of `References`.
  - Logs exist (`report.json`).
  - Save `scorecard.json` and return the dict.

### Please write your answer below

In [ ]:
# Q5 — Solution
def run_pipeline(topic: str) -> Dict[str, Any]:
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
def evaluate_outputs(brief_path: str = str(ARTIFACTS_DIR/'brief.md'), report_path: str = str(ARTIFACTS_DIR/'report.json')) -> Dict[str, Any]:
# YOUR CODE HERE
raise NotImplementedError()

---
### Make sure to write your answer above this line
 -  Feel free to add as many code cells above this line as you wish    
 -  Make sure to save your function in the variable named `run_pipeline`, `evaluate_outputs`.
 -  If the function is not found, you answer will be rejected by the system

In [ ]:
# Q5 — Public Tests
# BEGIN PUBLIC TESTS
_ = run_pipeline("Test topic")
sc = evaluate_outputs()
assert isinstance(sc, dict) and "has_references" in sc
# END PUBLIC TESTS

In [ ]:
# Q5 — Hidden Tests

---

## Submission

Submit this notebook plus the contents of the `artifacts/` folder:
- `brief.md`, `report.json`, `scorecard.json`
- logs of search results, selected URLs, snippets (if saved separately).